# DevOps AI Agent — ML Pipeline Training & Evaluation

PhD Research Notebook: Hybrid ML approach for autonomous DevOps observability analysis.

**Research Question:** Can an ensemble of specialised ML models match or exceed a large language model (Claude API) for real-time DevOps anomaly detection, forecasting, root cause analysis, and operational reporting?

**Models:**
1. Anomaly Detection — Isolation Forest + LSTM Autoencoder
2. Time-Series Forecasting — LSTM with Multi-head Attention
3. Root Cause Classification — XGBoost
4. Log Clustering — Sentence-BERT + UMAP + HDBSCAN
5. NLP Report Generation — Phi-3-mini (4-bit quantised)

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Project paths
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    DATA_DIR, MODEL_DIR, ARTIFACT_DIR, DEVICE,
    METRIC_FEATURES, ROOT_CAUSE_LABELS, SERVICES,
    anomaly_config, forecast_config, root_cause_config, log_cluster_config,
    training_config,
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print(f"Device: {DEVICE}")
print(f"Features: {METRIC_FEATURES}")
print(f"Services: {SERVICES}")

## 1. Data Generation & Exploration

In [ ]:
from data.generators.metric_generator import generate_training_dataset
from data.generators.log_generator import generate_log_dataset

# Generate training data
metrics_df, windows_df = generate_training_dataset(
    n_normal_windows=3000,
    n_anomaly_windows=1500,
    seed=42,
)
logs_df = generate_log_dataset(n_logs=20000, seed=42)

print(f"Metrics: {metrics_df.shape}")
print(f"Windows: {windows_df.shape}")
print(f"Logs: {logs_df.shape}")

In [ ]:
# Label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

windows_df['label'].value_counts().plot.bar(ax=axes[0], color=sns.color_palette('husl', len(windows_df['label'].unique())))
axes[0].set_title('Anomaly Type Distribution (Metric Windows)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

logs_df['severity'].value_counts().plot.bar(ax=axes[1], color=['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#8e44ad'])
axes[1].set_title('Log Severity Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / 'data_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample anomaly patterns visualisation
anomaly_types = ['memory_leak', 'cpu_saturation', 'downstream_timeout', 'request_spike']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, atype in zip(axes.flat, anomaly_types):
    window_id = windows_df[windows_df['label'] == atype]['window_id'].iloc[0]
    window = metrics_df[metrics_df['window_id'] == window_id]
    
    for feat in ['cpu_usage', 'error_rate', 'latency_p99']:
        vals = window[feat].values
        ax.plot(vals / vals.max() if vals.max() > 0 else vals, label=feat, alpha=0.8)
    
    ax.set_title(f'Anomaly: {atype}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('Time step')
    ax.set_ylabel('Normalised value')

plt.suptitle('Sample Anomaly Patterns (Normalised)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / 'anomaly_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

window_ids = windows_df['window_id'].values
labels = windows_df['label'].values

train_ids, temp_ids = train_test_split(window_ids, test_size=0.30, stratify=labels, random_state=42)
temp_labels = windows_df[windows_df['window_id'].isin(temp_ids)]['label'].values
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, stratify=temp_labels, random_state=42)

splits = {
    'train': {'windows': windows_df[windows_df['window_id'].isin(train_ids)],
              'metrics': metrics_df[metrics_df['window_id'].isin(train_ids)]},
    'val':   {'windows': windows_df[windows_df['window_id'].isin(val_ids)],
              'metrics': metrics_df[metrics_df['window_id'].isin(val_ids)]},
    'test':  {'windows': windows_df[windows_df['window_id'].isin(test_ids)],
              'metrics': metrics_df[metrics_df['window_id'].isin(test_ids)]},
}

for name, data in splits.items():
    print(f"{name}: {len(data['windows'])} windows, {len(data['metrics'])} samples")
    print(f"  Labels: {data['windows']['label'].value_counts().to_dict()}")

## 3. Model Training

### 3.1 Anomaly Detection (Isolation Forest + LSTM Autoencoder)

In [ ]:
from models.anomaly.detector import AnomalyDetector

detector = AnomalyDetector()

# Train Isolation Forest
if_results = detector.train_isolation_forest(splits['train']['windows'])
print(f"\nIsolation Forest — P: {if_results['precision']:.3f}, R: {if_results['recall']:.3f}, F1: {if_results['f1']:.3f}")

# Train LSTM Autoencoder
lstm_results = detector.train_lstm_autoencoder(
    splits['train']['metrics'],
    splits['val']['metrics'],
)
print(f"LSTM-AE — P: {lstm_results['precision']:.3f}, R: {lstm_results['recall']:.3f}, F1: {lstm_results['f1']:.3f}, AUC: {lstm_results['auc']:.3f}")

detector.save()

### 3.2 Time-Series Forecasting (LSTM + Attention)

In [ ]:
from models.forecasting.forecaster import MetricForecaster

forecaster = MetricForecaster()
forecast_results = forecaster.train(
    splits['train']['metrics'],
    splits['val']['metrics'],
)
forecaster.save()

print(f"\nForecaster Results:")
for k, v in forecast_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

### 3.3 Root Cause Classification (XGBoost)

In [ ]:
from models.root_cause.classifier import RootCauseClassifier

classifier = RootCauseClassifier()
rc_results = classifier.train(
    splits['train']['windows'],
    splits['train']['metrics'],
)
classifier.save()

# Feature importance plot
top_features = dict(list(classifier.feature_importance.items())[:15])
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(list(top_features.keys())[::-1], list(top_features.values())[::-1])
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Features — Root Cause Classification')
plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Log Clustering (Sentence-BERT + HDBSCAN)

In [ ]:
from models.log_clustering.clusterer import LogClusterer

clusterer = LogClusterer()
cluster_results = clusterer.train(logs_df)
clusterer.save()

print(f"\nClustering Results:")
for k, v in cluster_results.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Pattern summary
summary = clusterer.get_pattern_summary()
print(f"\nDiscovered {summary['total_patterns']} patterns")
for p in summary['patterns'][:5]:
    print(f"  Cluster {p['cluster_id']} ({p['size']} logs): {p['label']}")

## 4. Test Set Evaluation

In [ ]:
from evaluation.benchmark import ModelEvaluator

evaluator = ModelEvaluator()

# ── Anomaly Detection on test set ──
test_windows = splits['test']['windows']
test_metrics = splits['test']['metrics']

y_true, y_pred, y_scores = [], [], []
for wid in test_windows['window_id'].unique():
    window = test_metrics[test_metrics['window_id'] == wid]
    if len(window) >= 30:
        result = detector.predict(window)
        y_true.append(int(test_windows[test_windows['window_id'] == wid]['is_anomaly'].iloc[0]))
        y_pred.append(int(result['is_anomaly']))
        y_scores.append(result['anomaly_score'])

anomaly_eval = evaluator.evaluate_anomaly_detector(
    np.array(y_true), np.array(y_pred), np.array(y_scores)
)
print("Anomaly Detection (Test Set):")
for k, v in anomaly_eval.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# ── Root Cause Classification on test set ──
anomaly_test = test_windows[test_windows['label'] != 'normal']
rc_true, rc_pred, rc_conf = [], [], []

for _, row in anomaly_test.iterrows():
    window = test_metrics[test_metrics['window_id'] == row['window_id']]
    if len(window) > 0:
        pred = classifier.predict(window)
        rc_true.append(row['label'])
        rc_pred.append(pred['predicted_cause'])
        rc_conf.append(pred['confidence'])

from sklearn.metrics import classification_report, confusion_matrix
print("Root Cause Classification (Test Set):")
print(classification_report(rc_true, rc_pred))

# Confusion matrix
unique_labels = sorted(set(rc_true + rc_pred))
cm = confusion_matrix(rc_true, rc_pred, labels=unique_labels)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=unique_labels, yticklabels=unique_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Root Cause Classification — Confusion Matrix')
plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Latency Benchmarking

In [ ]:
from evaluation.benchmark import LatencyBenchmark

bench = LatencyBenchmark()

# Prepare a sample window
sample_wid = test_windows['window_id'].iloc[0]
sample_window = test_metrics[test_metrics['window_id'] == sample_wid]

# Benchmark each model
latency_results = {}
latency_results['anomaly_detection'] = bench.measure(detector.predict, sample_window, n_runs=50)
latency_results['forecasting'] = bench.measure(forecaster.predict, sample_window, n_runs=50)
latency_results['root_cause'] = bench.measure(classifier.predict, sample_window, n_runs=50)

print("\nLatency Benchmark Results:")
print(f"{'Model':<25} {'Mean (ms)':<12} {'P95 (ms)':<12} {'P99 (ms)':<12}")
print("-" * 60)
for name, res in latency_results.items():
    print(f"{name:<25} {res['mean_ms']:<12.2f} {res['p95_ms']:<12.2f} {res['p99_ms']:<12.2f}")

## 6. Cost Analysis — ML vs Claude API

In [ ]:
from evaluation.benchmark import CostAnalyser

cost = CostAnalyser()
comparison = cost.compare(
    avg_input_tokens=2000,
    avg_output_tokens=800,
    calls_per_day=288,  # every 5 minutes
)

print("\nCost Comparison (Monthly):")
print(f"  Claude API:  ${comparison['api_cost']['monthly_total_cost_usd']:.2f}/month")
print(f"  ML Pipeline: ${comparison['ml_cost']['monthly_cost_usd']:.2f}/month")
print(f"  Savings:     ${comparison['monthly_savings_usd']:.2f}/month ({comparison['savings_percentage']:.1f}%)")
print(f"\n  GPU utilisation: {comparison['ml_cost']['gpu_utilisation_pct']:.2f}%")

## 7. Statistical Significance Testing

In [ ]:
from evaluation.benchmark import StatisticalTests

tests = StatisticalTests()

# Bootstrap CI for anomaly detection F1
# (In practice, collect per-sample scores from multiple runs)
anomaly_scores = np.array(y_scores)
ci = tests.bootstrap_ci(anomaly_scores, n_bootstrap=10000, confidence=0.95)
print(f"\nAnomaly Score 95% CI: [{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]")
print(f"Mean: {ci['mean']:.4f} ± {ci['std']:.4f}")

## 8. Results Summary Table (Publication-Ready)

In [ ]:
# Compile all results into a summary table
results_table = pd.DataFrame([
    {'Model': 'Anomaly Detection (IF+LSTM)', 
     'Precision': anomaly_eval.get('precision', 0),
     'Recall': anomaly_eval.get('recall', 0),
     'F1': anomaly_eval.get('f1_score', 0),
     'AUC-ROC': anomaly_eval.get('auc_roc', 0),
     'Latency (ms)': latency_results.get('anomaly_detection', {}).get('mean_ms', 0),
    },
    {'Model': 'Forecaster (LSTM+Attn)',
     'MAE': forecast_results.get('train_mae', 0),
     'RMSE': forecast_results.get('train_rmse', 0),
     'Latency (ms)': latency_results.get('forecasting', {}).get('mean_ms', 0),
    },
    {'Model': 'Root Cause (XGBoost)',
     'Accuracy': rc_results.get('train_accuracy', 0),
     'F1 (weighted)': rc_results.get('train_f1_weighted', 0),
     'CV F1': rc_results.get('cv_f1_mean', 0),
     'Latency (ms)': latency_results.get('root_cause', {}).get('mean_ms', 0),
    },
    {'Model': 'Log Clustering (SBERT+HDBSCAN)',
     'Silhouette': cluster_results.get('silhouette_score', 0),
     'ARI': cluster_results.get('adjusted_rand_index', 0),
     'NMI': cluster_results.get('normalised_mutual_info', 0),
     'Clusters': cluster_results.get('n_clusters', 0),
    },
])

print("\n" + "=" * 70)
print("  RESULTS SUMMARY")
print("=" * 70)
print(results_table.to_string(index=False))

# Save
results_table.to_csv(str(ARTIFACT_DIR / 'results_summary.csv'), index=False)
print(f"\nSaved to {ARTIFACT_DIR / 'results_summary.csv'}")

In [ ]:
# Save all results as JSON artifact for reproducibility
all_results = {
    'anomaly_detection': {
        'isolation_forest': if_results,
        'lstm_autoencoder': lstm_results,
        'test_evaluation': anomaly_eval,
        'latency': latency_results.get('anomaly_detection', {}),
    },
    'forecasting': {
        'training': forecast_results,
        'latency': latency_results.get('forecasting', {}),
    },
    'root_cause': {
        'training': rc_results,
        'latency': latency_results.get('root_cause', {}),
    },
    'log_clustering': cluster_results,
    'cost_analysis': comparison,
}

with open(ARTIFACT_DIR / 'full_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"Full results saved to {ARTIFACT_DIR / 'full_results.json'}")